# 01 — Collect a Training Dataset

This notebook collects transition data from Procedural Frozen Lake and pushes it to the Hugging Face Hub so the same rollout data can be reused across training runs.

In [16]:
import torch

from mouse_envs import EnvConfig, make_env
from mouse_core.data import Datastore, push_stores_to_hub

DATASET_ID  = "mouse-example-dataset"
TOTAL_STEPS = 1500
NUM_ENVS = 1000

## Build Environment

`EnvConfig` describes the environment you want to collect from. `make_env` instantiates it and handles all the Gymnasium wiring. See the `mouse-env` package for more information about available environments and configuration options.

Key parameters for this run:

- **One config per env instance** - `NUM_ENVS` Procedural Frozen Lake env instances are listed explicitly in `configs`. To add or remove envs, edit `NUM_ENVS`.
- **`reset_seed=i`** - each env instance gets its own reset stream.
- **`map_seed=i`** - each env instance gets a deterministic Procedural Frozen Lake map.
- **`episodes_per_task=0`** - each env instance runs one infinite task, so episode boundaries do not reset the policy context.
- **`is_slippery=False`** - actions are deterministic, which keeps the oracle labels and learned behavior easy to inspect.
- **`min_width=4, max_width=8, min_height=4, max_height=8`** - maps can be as large as 8x8, matching the 64-observation vocab used by the training examples.
- **`max_episode_steps=50`** - episodes truncate after 50 steps if the agent has not reached a terminal state.
- **`q_star_source={"provider": "env_q_star"}`** - runs value iteration for the current map and attaches the exact optimal Q-values to every step output as `info_q_star`.

In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "is_slippery": False,
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "step_penalty": -0.01,
            "max_episode_steps": 50,
            "map_seed": i,
        },
        q_star_source={"provider": "env_q_star"},  # attached to every step as info_q_star
    )
    for i in range(NUM_ENVS)
]

env = make_env(configs)
print("Environments:", env.names)

Environments: ('proc_frozenlake_0', 'proc_frozenlake_1', 'proc_frozenlake_2', 'proc_frozenlake_3', 'proc_frozenlake_4', 'proc_frozenlake_5', 'proc_frozenlake_6', 'proc_frozenlake_7', 'proc_frozenlake_8', 'proc_frozenlake_9', 'proc_frozenlake_10', 'proc_frozenlake_11', 'proc_frozenlake_12', 'proc_frozenlake_13', 'proc_frozenlake_14', 'proc_frozenlake_15', 'proc_frozenlake_16', 'proc_frozenlake_17', 'proc_frozenlake_18', 'proc_frozenlake_19', 'proc_frozenlake_20', 'proc_frozenlake_21', 'proc_frozenlake_22', 'proc_frozenlake_23', 'proc_frozenlake_24', 'proc_frozenlake_25', 'proc_frozenlake_26', 'proc_frozenlake_27', 'proc_frozenlake_28', 'proc_frozenlake_29', 'proc_frozenlake_30', 'proc_frozenlake_31', 'proc_frozenlake_32', 'proc_frozenlake_33', 'proc_frozenlake_34', 'proc_frozenlake_35', 'proc_frozenlake_36', 'proc_frozenlake_37', 'proc_frozenlake_38', 'proc_frozenlake_39', 'proc_frozenlake_40', 'proc_frozenlake_41', 'proc_frozenlake_42', 'proc_frozenlake_43', 'proc_frozenlake_44', 'proc

## Create Datastores

A `Datastore` is a sequential container for step data backed by a Hugging Face Arrow dataset. Each row is a plain Python dict — no fixed schema required. Whatever fields your environment returns, you can store them as-is.

The `name` you give a store becomes its config identifier on the Hub, so each parallel env instance can be fetched back by name.

In [18]:
stores = [Datastore(name=name) for name in env.names]  # one store per parallel env instance

## Collect transitions

Each step merges the input we sent (`{"action": ...}`) with the environment's output (`observation`, `reward`, `done`, `info_q_star`, ...) into a single dict and appends it to the matching store.

**Oracle ramp:** at step `t`, `oracle_prob = t / (TOTAL_STEPS - 1)`. Each env instance independently samples: with probability `oracle_prob` it picks `argmax(info_q_star)` (the optimal action); otherwise it picks a random action. This means the first steps explore freely while later steps demonstrate expert behavior.

**Initial reset frame:** the very first `env.step` triggers the environment reset; the env ignores the action and returns the initial observation. We store this frame so every datastore starts from timestep 0 and is time-aligned.

In [19]:
def pick_inputs(outputs, env, oracle_prob: float) -> list[dict]:
    random_inputs = env.sample_random_inputs()
    inputs = []
    for i, out in enumerate(outputs):
        if torch.rand(1).item() >= oracle_prob:  # explore
            inputs.append(random_inputs[i])
        else:
            action = out.get("info_q_star").argmax()  # best action per Q*
            inputs.append({"action": torch.tensor(action, dtype=torch.int64)})
    return inputs

outputs = None
env.tracker.clear()
for t in range(TOTAL_STEPS):
    oracle_prob = t / (TOTAL_STEPS - 1)         # 0.0 → 1.0 over the run
    if outputs is None:                          # first step triggers reset
        inputs = env.sample_random_inputs()
    else:
        inputs = pick_inputs(outputs, env, oracle_prob)
    outputs = env.step(inputs)
    for store, inp, out in zip(stores, inputs, outputs):
        store.append({**inp, **out})             # merge input + output into one row

env.close()

for name, rewards, lengths in zip(env.names, env.tracker.episode_cum_rewards, env.tracker.episode_lengths):
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).mean().item() if n else float("nan")
    print(f"{name}: {n} episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}")

proc_frozenlake_0: 186 episodes | mean reward 0.478 | mean length 7.1
proc_frozenlake_1: 149 episodes | mean reward 0.775 | mean length 9.0
proc_frozenlake_2: 204 episodes | mean reward 0.647 | mean length 6.3
proc_frozenlake_3: 166 episodes | mean reward 0.468 | mean length 8.0
proc_frozenlake_4: 203 episodes | mean reward 0.503 | mean length 6.4
proc_frozenlake_5: 162 episodes | mean reward 0.819 | mean length 8.2
proc_frozenlake_6: 134 episodes | mean reward 0.272 | mean length 10.2
proc_frozenlake_7: 255 episodes | mean reward 0.524 | mean length 4.9
proc_frozenlake_8: 197 episodes | mean reward 0.640 | mean length 6.6
proc_frozenlake_9: 266 episodes | mean reward 0.615 | mean length 4.6
proc_frozenlake_10: 277 episodes | mean reward 0.234 | mean length 4.4
proc_frozenlake_11: 219 episodes | mean reward 0.480 | mean length 5.8
proc_frozenlake_12: 213 episodes | mean reward 0.405 | mean length 6.0
proc_frozenlake_13: 262 episodes | mean reward 0.648 | mean length 4.7
proc_frozenlake

## Push to the Hub

Upload all four stores to your Hugging Face account as a single dataset repo. Each store is saved as a separate named config, which keeps the parallel env streams distinct when the dataset is loaded later.

In [20]:
url = push_stores_to_hub(stores, repo_id=DATASET_ID, split="train", private=False)
print(f"Pushed to {url}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-example-dataset (proc_frozenlake_0/train: 1500, proc_frozenlake_1/train: 1500, proc_frozenlake_2/train: 1500, proc_frozenlake_3/train: 1500, proc_frozenlake_4/train: 1500, proc_frozenlake_5/train: 1500, proc_frozenlake_6/train: 1500, proc_frozenlake_7/train: 1500, proc_frozenlake_8/train: 1500, proc_frozenlake_9/train: 1500, proc_frozenlake_10/train: 1500, proc_frozenlake_11/train: 1500, proc_frozenlake_12/train: 1500, proc_frozenlake_13/train: 1500, proc_frozenlake_14/train: 1500, proc_frozenlake_15/train: 1500, proc_frozenlake_16/train: 1500, proc_frozenlake_17/train: 1500, proc_frozenlake_18/train: 1500, proc_frozenlake_19/train: 1500, proc_frozenlake_20/train: 1500, proc_frozenlake_21/train: 1500, proc_frozenlake_22/train: 1500, proc_frozenlake_23/train: 1500, proc_frozenlake_24/train: 1500, proc_frozenlake_25/train: 1500, proc_frozenlake_26/train: 1500, proc_frozenlake_27/train: 1500, proc_frozenlake_28/train: 1500, proc_frozenlake_29/train: 1500, proc_fr